### Chroma
Chroma is a AI-native open-source vector database focused on developer productivity and happiness. Chroma is licensed under Apache 2.0.

https://docs.langchain.com/oss/python/langchain/overview

#### Building a sample vector db

In [2]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# Load

loader = TextLoader("speech.txt")
data = loader.load()
data

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\nâ€¦\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness 

In [ ]:
# Split

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
splits = text_splitter.split_documents(data)

In [9]:
# Embed

embedding = OllamaEmbeddings(model="gemma:2b")
vectordb = Chroma.from_documents(splits, embedding)
vectordb

In [10]:
# Query

query = "What does the speaker believe is the main reason the United States should enter the war?"
docs = vectordb.similarity_search(query)
docs[0].page_content

'democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

In [11]:
# Save locally

vectordb = Chroma.from_documents(
    splits, 
    embedding,
    persist_directory="./chroma_db"
)

In [13]:
# Load from local

db2 = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embedding
)
docs = db2.similarity_search(query)
print(docs[0].page_content)

democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.


In [14]:
#  Similarity search with score

docs = vectordb.similarity_search_with_score(query)
docs

[(Document(id='0a41dc6e-cfa5-4685-9a34-24f3c1be73c9', metadata={'source': 'speech.txt'}, page_content='democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
  3511.509765625),
 (Document(id='528ced43-ba80-4d3e-8b0a-419936baf724', metadata={'source': 'speech.txt'}, page_content='democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
  3511.509765625),
 (Document(id='f1c484dd-36bf-4911-9aab-be54f4fb2541', metadata={'source': 'speech.txt'}, page_content='government in the hour of test. They are, m

In [15]:
# Retriever Option

retriever = vectordb.as_retriever()
retriever.invoke(query)[0].page_content

'democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'